In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))


# Baseline — val-loss early stopping (multi-seed)

`roberta-base` fine-tuned on the train split, early stopping on val loss (patience 3). Seeds 42, 1, 2.

In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from huggingface_hub import snapshot_download
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset

DATA_DIR = Path(snapshot_download(repo_id='tamarasuarezrod/longeval-data', repo_type='dataset'))

MODEL_NAME = 'roberta-base'
SEEDS      = [42, 1, 2]
STRATEGY   = 'baseline-valloss'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    cap = torch.cuda.get_device_capability()
    assert cap[0] >= 7, f'GPU sm_{cap[0]}{cap[1]} not supported'
    print('GPU:', torch.cuda.get_device_name())


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Device: cuda
GPU: Tesla T4


## Data — load once

In [3]:
def load_split(path, label_col='label'):
    with open(path) as f:
        records = json.load(f)
    df = pd.DataFrame(records).rename(columns={label_col: 'label'})
    return df[['pp_text', 'label']]

splits = {
    'train':       load_split(DATA_DIR / 'train_eval/train.json',             label_col='distant_label'),
    'eval':        load_split(DATA_DIR / 'train_eval/interim_eval_2016.json', label_col='distant_label'),
    'test_within': load_split(DATA_DIR / 'test/interim_test_2016.json'),
    'test_short':  load_split(DATA_DIR / 'test/interim_test_2018.json'),
    'test_long':   load_split(DATA_DIR / 'test/interim_test_2021.json'),
}
for name, df in splits.items():
    print(f'{name}: {len(df)} rows')

label2id = {l: i for i, l in enumerate(sorted(splits['train']['label'].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Labels:', label2id)


train: 49608 rows
eval: 1344 rows
test_within: 908 rows
test_short: 908 rows
test_long: 908 rows
Labels: {'negative': 0, 'positive': 1}


## Tokenise — once

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    d = Dataset.from_dict({'text':  df['pp_text'].tolist(),
                           'label': df['label'].map(label2id).tolist()})
    return d.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=128),
                 batched=True, remove_columns=['text'])

train_ds = make_dataset(splits['train'])
eval_ds  = make_dataset(splits['eval'])
test_ds  = {n: make_dataset(df) for n, df in splits.items() if n not in ('train', 'eval')}
print('Train:', len(train_ds), ' Eval:', len(eval_ds))


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1344 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Map:   0%|          | 0/908 [00:00<?, ? examples/s]

Train: 49608  Eval: 1344


## Training loop (seeds 42, 1, 2)

In [5]:
all_results  = {}
saved_models = {}

for SEED in SEEDS:
    print(f'\n{"="*50}  SEED {SEED}')
    set_seed(SEED)
    models_dir = Path(f'/tmp/baseline_seed{SEED}')
    models_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=str(models_dir), num_train_epochs=25,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=2e-05, warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        logging_steps=50, fp16=(DEVICE=='cuda'), seed=SEED, report_to='none')

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=eval_ds,
                      processing_class=tokenizer,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()

    seed_results = {}
    for name, ds in test_ds.items():
        preds_out = trainer.predict(ds)
        f1 = f1_score(np.array(ds['label']), np.argmax(preds_out.predictions, axis=1), average='macro')
        seed_results[name] = f1
        print(f'  {name}: F1={f1:.4f}')

    all_results[SEED]  = seed_results
    saved_models[SEED] = trainer.model



==================================================  SEED 42


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/di

Epoch,Training Loss,Validation Loss
1,0.932407,0.964375
2,0.859714,0.963725
3,0.847724,0.889641
4,0.644026,1.012356
5,0.547143,1.016517
6,0.444487,1.227843


...

  test_within: F1=0.7087


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_short: F1=0.6719


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


  test_long: F1=0.6828


## Summary

In [6]:
import numpy as np

print(f"{'seed':>8}  {'within':>7}  {'short':>7}  {'long':>7}  {'RPD_short':>10}  {'RPD_long':>9}")
for seed, r in all_results.items():
    rpd_s = (r['test_short']  - r['test_within']) / r['test_within']
    rpd_l = (r['test_long']   - r['test_within']) / r['test_within']
    print(f"{seed:>8}  {r['test_within']:>7.4f}  {r['test_short']:>7.4f}  {r['test_long']:>7.4f}  {rpd_s:>+10.4f}  {rpd_l:>+9.4f}")

print()
for sp in ['test_within', 'test_short', 'test_long']:
    vals = [all_results[s][sp] for s in all_results]
    print(f"{sp}: mean={np.mean(vals):.4f}  std={np.std(vals,ddof=1):.4f}  [{min(vals):.4f}–{max(vals):.4f}]")


    seed   within    short     long   RPD_short   RPD_long
      42   0.7298   0.6770   0.6880     -0.0723    -0.0573
       1   0.7018   0.6643   0.6865     -0.0535    -0.0218
       2   0.7087   0.6719   0.6828     -0.0519    -0.0365

test_within: mean=0.7134  std=0.0146  [0.7018–0.7298]
test_short: mean=0.6711  std=0.0064  [0.6643–0.6770]
test_long: mean=0.6858  std=0.0027  [0.6828–0.6880]


## Save to HuggingFace

In [7]:
for seed, model in saved_models.items():
    repo = f'tamarasuarezrod/longeval-{STRATEGY}-seed{seed}'
    model.push_to_hub(repo)
    tokenizer.push_to_hub(repo)
    print('Saved:', repo)


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved: tamarasuarezrod/longeval-baseline-valloss-seed2
